In [6]:
## claude v2

import pandas as pd
import numpy as np
import os

# --- Configuration ---
N_ROWS = 10000
FILE_NAME = 'realistic_migraine_data_v2.csv'
np.random.seed(42)

# --- 1. Initialize Dataset & Generate Demographic Features ---
data = {}

# Gender (0=Female, 1=Male): Skewed toward Female (~75% F : 25% M)
data['Gender'] = np.random.choice([0, 1], N_ROWS, p=[0.75, 0.25])
is_female = data['Gender'] == 0

# Age: Peak incidence around 35
base_age = np.random.normal(loc=37, scale=10, size=N_ROWS)
data['Age'] = np.clip(base_age, 18, 65).astype(int)

# Menstruation
is_reproductive_age_female = (is_female) & (data['Age'] >= 18) & (data['Age'] <= 50)
menstruation_potential = np.random.choice([0, 1], N_ROWS, p=[0.80, 0.20])
data['Menstruation'] = np.where(is_reproductive_age_female, menstruation_potential, 0)

# --- 2. Lifestyle / Trigger Features ---

# Stress Level (1–10) with more variance
data['Stress_Level'] = np.clip(
    np.round(np.random.normal(loc=6.0, scale=2.5, size=N_ROWS)),
    1, 10
).astype(int)

# Sleep Duration (hours) - more realistic variance
data['Sleep_Duration'] = np.clip(
    np.random.normal(loc=6.8, scale=1.5, size=N_ROWS),
    3.5, 10.0
).round(1)

data['No_of_Meals'] = np.random.randint(1, 6, N_ROWS)
data['Water_Intake'] = np.random.randint(1, 6, N_ROWS)

# --- NEW FEATURES for more complexity ---

# Caffeine intake (cups per day: 0-8)
data['Caffeine_Intake'] = np.random.poisson(lam=2.5, size=N_ROWS).clip(0, 8)

# Exercise (days per week: 0-7)
data['Exercise_Days'] = np.random.binomial(n=7, p=0.35, size=N_ROWS)

# Screen time (hours per day: 2-16)
data['Screen_Time'] = np.clip(
    np.random.gamma(shape=3, scale=2.5, size=N_ROWS),
    2, 16
).round(1)

# --- 3. INDIVIDUAL SENSITIVITY PROFILES ---
# Each person has unique trigger sensitivities (KEY IMPROVEMENT)

stress_sensitivity = np.random.beta(2, 5, N_ROWS) * 4  # 0-4 range
sleep_sensitivity = np.random.beta(2, 5, N_ROWS) * 3
caffeine_sensitivity = np.random.beta(2, 8, N_ROWS) * 3
weather_sensitivity = np.random.beta(2, 6, N_ROWS) * 2

# --- 4. NON-LINEAR TRIGGER EFFECTS ---

# Stress: Non-linear with threshold effects
stress_effect = np.where(
    data['Stress_Level'] > 7,
    stress_sensitivity * (data['Stress_Level'] - 7) ** 1.5,  # Exponential above threshold
    stress_sensitivity * (data['Stress_Level'] / 10)
)

# Sleep: Both too little AND too much can trigger (U-shaped)
optimal_sleep = 7.5
sleep_deviation = np.abs(data['Sleep_Duration'] - optimal_sleep)
sleep_effect = sleep_sensitivity * (sleep_deviation ** 1.3)

# Caffeine: Withdrawal or excess both bad
caffeine_effect = np.where(
    (data['Caffeine_Intake'] == 0) | (data['Caffeine_Intake'] > 5),
    caffeine_sensitivity * 1.5,
    caffeine_sensitivity * 0.3
)

# Dehydration effect (non-linear)
dehydration_effect = np.where(
    data['Water_Intake'] < 3,
    (3 - data['Water_Intake']) ** 1.8,
    0
)

# Menstrual effect (with individual variation)
menstrual_effect = data['Menstruation'] * np.random.uniform(1.5, 4.0, N_ROWS)

# --- 5. INTERACTION EFFECTS (KEY IMPROVEMENT) ---

# Stress + Poor sleep = multiplicative effect
stress_sleep_interaction = (
    (data['Stress_Level'] / 10) * 
    (sleep_deviation / 3.5) * 
    np.random.uniform(0.5, 2.5, N_ROWS)
)

# Screen time + Light sensitivity interaction
screen_light_boost = (data['Screen_Time'] / 16) * np.random.uniform(0, 2, N_ROWS)

# Skipped meals + Low water interaction
meal_water_penalty = np.where(
    (data['No_of_Meals'] <= 2) & (data['Water_Intake'] <= 2),
    np.random.uniform(1.5, 3.5, N_ROWS),
    0
)

# --- 6. Weather Features (one-hot encoded) ---
weather_options = ['Sunny', 'Cloudy', 'Rainy', 'Snowy']
weather_probs = [0.4, 0.35, 0.2, 0.05]

weather_choices = np.random.choice(weather_options, N_ROWS, p=weather_probs)
weather_df = pd.DataFrame(0, index=range(N_ROWS), columns=[f'Weather_{w}' for w in weather_options])

for i, choice in enumerate(weather_choices):
    weather_df.loc[i, f'Weather_{choice}'] = 1

# Weather effect (with individual sensitivity)
weather_effect = (
    (weather_df['Weather_Rainy'] * 1.2 + 
     weather_df['Weather_Snowy'] * 0.8) * 
    weather_sensitivity
)

# --- 7. Calculate Pain Severity with MORE COMPLEXITY ---

# Base pain from all factors (NOT weighted sum, but complex interactions)
base_pain_score = (
    stress_effect * 1.2 +
    sleep_effect * 1.0 +
    caffeine_effect * 0.8 +
    dehydration_effect * 0.7 +
    menstrual_effect * 0.9 +
    stress_sleep_interaction * 1.5 +  # Interaction effects
    screen_light_boost * 0.6 +
    meal_water_penalty * 0.8 +
    weather_effect * 0.5 +
    np.random.normal(0, 2.5, N_ROWS)  # HIGH random noise
)

# Add outliers (some people just have bad days)
outlier_mask = np.random.rand(N_ROWS) < 0.05
base_pain_score[outlier_mask] += np.random.uniform(3, 6, outlier_mask.sum())

data['Pain_Severity'] = np.clip(np.round(base_pain_score), 1, 10).astype(int)

# --- 8. Sensitivity Features (LESS correlated with pain) ---

# Light sensitivity: partially correlated but with noise
light_base_prob = (data['Pain_Severity'] / 10) * 0.6 + 0.15
data['Light_Sensitivity'] = (np.random.rand(N_ROWS) < light_base_prob).astype(int)

# Add trait-based light sensitivity (some people are always sensitive)
trait_sensitive = np.random.rand(N_ROWS) < 0.15
data['Light_Sensitivity'] = np.where(trait_sensitive, 1, data['Light_Sensitivity'])

# Noise sensitivity: similar approach
noise_base_prob = (data['Pain_Severity'] / 10) * 0.5 + 0.2
data['Noise_Sensitivity'] = (np.random.rand(N_ROWS) < noise_base_prob).astype(int)

# --- 9. Combine all data ---
df = pd.DataFrame(data)
df = pd.concat([df, weather_df], axis=1)

# --- 10. Migraine Score with REDUCED DETERMINISM ---

# Normalize features with MORE NOISE
norm_stress = df['Stress_Level'] / 10 + np.random.normal(0, 0.15, N_ROWS)
norm_sleep = (9.5 - df['Sleep_Duration']) / 5.5 + np.random.normal(0, 0.2, N_ROWS)
norm_menstruation = df['Menstruation'] * np.random.uniform(0.7, 1.3, N_ROWS)
norm_pain = df['Pain_Severity'] / 10 + np.random.normal(0, 0.15, N_ROWS)
norm_light = df['Light_Sensitivity'] * np.random.uniform(0.5, 1.5, N_ROWS)
norm_noise = df['Noise_Sensitivity'] * np.random.uniform(0.5, 1.5, N_ROWS)

# LESS predictable weights (vary by person)
weight_variation = np.random.normal(1.0, 0.25, N_ROWS)

migraine_score = (
    0.30 * norm_pain * weight_variation +
    0.15 * norm_stress * np.random.uniform(0.8, 1.2, N_ROWS) +
    0.15 * norm_sleep * np.random.uniform(0.8, 1.2, N_ROWS) +
    0.10 * norm_light * np.random.uniform(0.6, 1.4, N_ROWS) +
    0.10 * norm_noise * np.random.uniform(0.6, 1.4, N_ROWS) +
    0.08 * norm_menstruation +
    0.12 * np.random.uniform(0, 1, N_ROWS)  # Pure randomness (unobserved factors)
)

df['Migraine_Score'] = (migraine_score * 100).clip(0, 100).round(2)

# --- 11. Add SYSTEMATIC BIASES & MEASUREMENT ERROR ---

# Age-based underreporting
age_bias = np.where(df['Age'] < 30, 0.9, np.where(df['Age'] > 55, 0.85, 1.0))
df['Migraine_Score'] *= age_bias

# Gender reporting differences (women may report slightly higher)
gender_bias = np.where(df['Gender'] == 0, np.random.uniform(1.0, 1.08, N_ROWS), 1.0)
df['Migraine_Score'] *= gender_bias

# Random measurement error (+/- 8%)
measurement_error = np.random.normal(1.0, 0.08, N_ROWS)
df['Migraine_Score'] *= measurement_error

df['Migraine_Score'] = df['Migraine_Score'].clip(0, 100).round(2)

# --- 12. Add MISSING VALUES (realistic) ---
missing_rate = 0.03
for col in ['Sleep_Duration', 'Water_Intake', 'Caffeine_Intake', 'Screen_Time']:
    missing_idx = np.random.choice(df.index, size=int(N_ROWS * missing_rate), replace=False)
    df.loc[missing_idx, col] = np.nan

# Stress level sometimes not reported
missing_idx = np.random.choice(df.index, size=int(N_ROWS * 0.02), replace=False)
df.loc[missing_idx, 'Stress_Level'] = np.nan

# --- 13. Add TEMPORAL PATTERNS (if dataset had time component) ---
# This adds autocorrelation - some days naturally cluster
cluster_effect = np.random.normal(0, 5, N_ROWS // 50).repeat(50)[:N_ROWS]
df['Migraine_Score'] += cluster_effect
df['Migraine_Score'] = df['Migraine_Score'].clip(0, 100).round(2)

# --- 14. Order columns ---
column_order = [
    'Age', 'Stress_Level', 'Gender', 'Menstruation', 'Sleep_Duration',
    'Pain_Severity', 'Light_Sensitivity', 'Noise_Sensitivity',
    'No_of_Meals', 'Water_Intake', 'Caffeine_Intake', 'Exercise_Days',
    'Screen_Time', 'Weather_Sunny', 'Weather_Cloudy',
    'Weather_Rainy', 'Weather_Snowy', 'Migraine_Score'
]
df = df[column_order]

# --- 15. Save ---
df.to_csv(FILE_NAME, index=False)
print(f"✅ Dataset '{FILE_NAME}' with {N_ROWS} rows generated.")
print(f"📊 Missing values: {df.isnull().sum().sum()} total")
print("\n--- Sample Data ---")
print(df.head(10))
print("\n--- Statistics ---")
print(df.describe().round(2))

C:\Users\hp\AppData\Local\Temp\ipykernel_21692\1950588386.py:72: RuntimeWarning: invalid value encountered in power
  stress_sensitivity * (data['Stress_Level'] - 7) ** 1.5,  # Exponential above threshold
C:\Users\hp\AppData\Local\Temp\ipykernel_21692\1950588386.py:91: RuntimeWarning: invalid value encountered in power
  (3 - data['Water_Intake']) ** 1.8,


✅ Dataset 'realistic_migraine_data_v2.csv' with 10000 rows generated.
📊 Missing values: 1400 total

--- Sample Data ---
   Age  Stress_Level  Gender  Menstruation  Sleep_Duration  Pain_Severity  \
0   22          10.0       0             0             4.3             10   
1   25           3.0       1             0             7.0              1   
2   40           8.0       0             0             4.6              7   
3   25          10.0       0             0             4.7             10   
4   48           1.0       0             0             5.3              2   
5   36           6.0       0             1             8.4              4   
6   37           8.0       0             0             7.6              6   
7   34           7.0       1             0             7.3              1   
8   44           7.0       0             0             5.5              5   
9   44           6.0       0             0             6.2              1   

   Light_Sensitivity  Noise_Sens